In [9]:
import pandas as pd
import os

# ==========================================
# 1. ĐỌC TẤT CẢ DỮ LIỆU THÔ
# ==========================================
df_ecb_rate = pd.read_csv("C:/Users/Viet Hung/Downloads/gois1-1109/Raw_ECB_PolicyRate.csv")
df_eur_usd = pd.read_csv("C:/Users/Viet Hung/Downloads/gois1-1109/Raw_ECB_EURUSD.csv")
df_inflation = pd.read_csv("C:/Users/Viet Hung/Downloads/gois1-1109/Raw_Eurostat_Inflation.csv")
df_unemp = pd.read_csv("C:/Users/Viet Hung/Downloads/gois1-1109/Raw_Eurostat_Unemployment.csv")
df_gdp = pd.read_csv("C:/Users/Viet Hung/Downloads/gois1-1109/Raw_Eurostat_GDP.csv") # Thêm dữ liệu GDP Eurostat
df_lending = pd.read_csv("C:/Users/Viet Hung/Downloads/gois1-1109/Raw_ECB_BankLendingRate.csv")
df_credit = pd.read_csv("C:/Users/Viet Hung/Downloads/gois1-1109/Raw_ECB_BankCreditGrowth.csv")

# ==========================================
# 2. XỬ LÝ NHÓM BIẾN TOÀN KHỐI (Ghép theo Date)
# ==========================================
df_ecb_rate = df_ecb_rate[['TIME_PERIOD', 'OBS_VALUE']].rename(columns={'TIME_PERIOD': 'Date', 'OBS_VALUE': 'Policy_Rate'})
df_ecb_rate['Date'] = pd.to_datetime(df_ecb_rate['Date']).dt.to_period('M')
df_ecb_rate = df_ecb_rate.groupby('Date').last().reset_index()

df_eur_usd = df_eur_usd[['TIME_PERIOD', 'OBS_VALUE']].rename(columns={'TIME_PERIOD': 'Date', 'OBS_VALUE': 'EUR_USD'})
df_eur_usd['Date'] = pd.to_datetime(df_eur_usd['Date']).dt.to_period('M')

df_lending = df_lending[['TIME_PERIOD', 'OBS_VALUE']].rename(columns={'TIME_PERIOD': 'Date', 'OBS_VALUE': 'Lending_Rate'})
df_lending['Date'] = pd.to_datetime(df_lending['Date']).dt.to_period('M')

df_ecb = pd.merge(df_ecb_rate, df_eur_usd, on='Date', how='outer')
df_ecb = pd.merge(df_ecb, df_lending, on='Date', how='outer')
df_ecb['Date'] = df_ecb['Date'].dt.to_timestamp() 

# ==========================================
# 3. XỬ LÝ NHÓM BIẾN QUỐC GIA (Ghép theo Country + Date)
# ==========================================
countries = ['DE', 'FR', 'IT', 'ES', 'NL']

def melt_eurostat(df, value_name):
    df = df.rename(columns={'geo\\TIME_PERIOD': 'Country', 'geo': 'Country'}, errors='ignore')
    df = df[df['Country'].isin(countries)]
    time_cols = [c for c in df.columns if str(c).startswith('20')]
    df_long = df.melt(id_vars=['Country'], value_vars=time_cols, var_name='Date', value_name=value_name)
    df_long['Date'] = pd.to_datetime(df_long['Date'], errors='coerce')
    return df_long.dropna(subset=['Date'])

# 3.1 Lạm phát & Thất nghiệp
df_inf_clean = melt_eurostat(df_inflation[(df_inflation['unit'] == 'RCH_A') & (df_inflation['coicop'] == 'CP00')], 'Inflation')
df_unemp_clean = melt_eurostat(df_unemp[(df_unemp['unit'] == 'PC_ACT') & (df_unemp['s_adj'] == 'SA') & (df_unemp['age'] == 'TOTAL')], 'Unemployment')

In [10]:
# 3.2 Tăng trưởng GDP thực tế từ Eurostat (Lọc đơn vị % thay đổi hoặc giá trị tùy ý, ở đây lấy CLV_PCH_SM hoặc tương đương nếu có, hoặc lọc theo na_item B1GQ)
df_gdp_clean = df_gdp[df_gdp['na_item'] == 'B1GQ'] # Lọc GDP tổng
df_gdp_clean = melt_eurostat(df_gdp_clean, 'GDP_Value')

# 3.3 Tăng trưởng tín dụng 
df_credit = df_credit[['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']].rename(columns={'REF_AREA': 'Country', 'TIME_PERIOD': 'Date', 'OBS_VALUE': 'Credit_Growth'})
df_credit = df_credit[df_credit['Country'].isin(countries)]
df_credit['Date'] = pd.to_datetime(df_credit['Date']).dt.to_period('M').dt.to_timestamp()

# -> Ghép các biến quốc gia lại với nhau
df_country = pd.merge(df_inf_clean, df_unemp_clean, on=['Country', 'Date'], how='outer')
df_country = pd.merge(df_country, df_gdp_clean, on=['Country', 'Date'], how='outer')
df_country = pd.merge(df_country, df_credit, on=['Country', 'Date'], how='outer')

# ==========================================
# 4. GHÉP BẢNG MASTER VÀ LỌC 2019-2026
# ==========================================
df_master = pd.merge(df_country, df_ecb, on='Date', how='left')
df_master = df_master[(df_master['Date'] >= '2019-01-01') & (df_master['Date'] <= '2026-12-31')]
df_master.sort_values(by=['Country', 'Date'], inplace=True)

# Lệnh Groupby giữ lại cột Country và lấp đầy dữ liệu thiếu (ffill cho cả GDP quý sang tháng)
df_master = df_master.groupby('Country', group_keys=False).apply(lambda x: x.ffill())

# ==========================================
# 5. XUẤT FILE DATASET
# ==========================================
os.makedirs("data_clean", exist_ok=True) 
df_master.to_csv("data_clean/master_dataset.csv", index=False)
print("Hoàn tất Merge! Bảng Master đã có cột Country và dữ liệu GDP Eurostat đầy đủ.")

Hoàn tất Merge! Bảng Master đã có cột Country và dữ liệu GDP Eurostat đầy đủ.


C:\Users\Viet Hung\AppData\Local\Temp\ipykernel_25836\4034100272.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_master = df_master.groupby('Country', group_keys=False).apply(lambda x: x.ffill())
